# Sudoku Solver

A sudoku puzzle is a 9x9 grid with some fixed values. To solve the puzzle, the empty cells of the grid must be filled so that the numbers 1 to 9 appear exactly once in each row, column, and 3x3 block of the grid.

Below is a sample puzzle along with its solution.

<img src="images/sudoku.png" style="width: 50%;"/>

This notebook implements a sudoku solver from scratch (NumPy only) based on an iterative backtracking depth-first search, and validates it on puzzles of four difficulty levels.

Conventions:
 * A puzzle is a 9x9 NumPy array where `0` marks an empty cell.
 * `sudoku_solver(sudoku)` returns the solved 9x9 array.
 * If the puzzle has no solution (e.g. the initial grid already contains a conflict), it returns a 9x9 array filled with `-1`.


## Loading the puzzle data

The cell below loads some sample sudoku puzzles to show the format. Puzzles come in four difficulty levels (very easy, easy, medium, hard): easier puzzles have more pre-filled cells. Each difficulty level comes with 15 puzzles, so a loaded file is a 15x9x9 array. Any level can be loaded by changing the file name.


In [1]:
import numpy as np

# Load sudokus
sudoku = np.load("data/very_easy_puzzle.npy")
print("very_easy_puzzle.npy has been loaded into the variable sudoku")
print(f"sudoku.shape: {sudoku.shape}, sudoku[0].shape: {sudoku[0].shape}, sudoku.dtype: {sudoku.dtype}")

# Load solutions for demonstration
solutions = np.load("data/very_easy_solution.npy")
print()

# Print the first 9x9 sudoku...
print("First sudoku:")
print(sudoku[0], "\n")

# ...and its solution
print("Solution of first sudoku:")
print(solutions[0])

very_easy_puzzle.npy has been loaded into the variable sudoku
sudoku.shape: (15, 9, 9), sudoku[0].shape: (9, 9), sudoku.dtype: int8

First sudoku:
[[1 0 4 3 8 2 9 5 6]
 [2 0 5 4 6 7 1 3 8]
 [3 8 6 9 5 1 4 0 2]
 [4 6 1 5 2 3 8 9 7]
 [7 3 8 1 4 9 6 2 5]
 [9 5 2 8 7 6 3 1 4]
 [5 2 9 6 3 4 7 8 1]
 [6 0 7 2 9 8 5 4 3]
 [8 4 3 0 1 5 2 6 9]] 

Solution of first sudoku:
[[1 7 4 3 8 2 9 5 6]
 [2 9 5 4 6 7 1 3 8]
 [3 8 6 9 5 1 4 7 2]
 [4 6 1 5 2 3 8 9 7]
 [7 3 8 1 4 9 6 2 5]
 [9 5 2 8 7 6 3 1 4]
 [5 2 9 6 3 4 7 8 1]
 [6 1 7 2 9 8 5 4 3]
 [8 4 3 7 1 5 2 6 9]]


## Implementation

The solver is made of the following functions:
* `sqr_extract`: checks whether a number already appears in the 3x3 block of a cell.
* `check_constraint`: checks the row, column and block constraints for a number in a cell.
* `check_initial_state`: validates the initial grid, detecting puzzles without a solution.
* `sudoku_solver`: the backtracking search.

**Approach.** The search visits the cells row by row. For each empty cell it computes the *frontier*, the list of values that satisfy the constraints given the cells filled so far, and assigns one of them (chosen at random), removing it from the frontier. When a cell has an empty frontier (a dead end), the algorithm backtracks to the previous empty cell that still has values to try, resetting the cells in between to `0`. If backtracking goes past the first cell, the puzzle has no solution and the `-1` array is returned. The original puzzle is kept in memory to know which cells are fixed. The frontier of a cell is only computed when empty, so a value already tried is not retried until the cells before it change.

In the worst case the time complexity is O(9^m), with m the number of empty cells; the extra space is O(1) in the grid size.


In [2]:
def sqr_extract(sudoku, row, pos, num):

    if pos % 3 == 0:
        range_pos = range(pos, pos +3)
    if (pos-1) % 3 == 0:
        range_pos = range(pos-1, pos + 2)
    if (pos+1) % 3 == 0:
        range_pos = range(pos-2, pos + 1)

    if row % 3 == 0:
        range_row = range(row, row +3)
    if (row-1) % 3 == 0:
        range_row = range(row-1, row + 2)
    if (row+1) % 3 == 0:
        range_row = range(row-2, row + 1)

    for r in range_row:
        for p in range_pos:
            if num == sudoku[r][p]:
                return False

    return True

def check_constraint(sudoku, row, pos, num = None):

    if not num:
        num_to_compare = sudoku[row][pos]
    else:
        num_to_compare = num

    for x in range(len(sudoku[row])):
        if num_to_compare == sudoku[row][x] and sudoku[row][x] != 0 and x != pos:
            return False

    for y in range(len(sudoku)):
        try:
            if num_to_compare == sudoku[y][pos] and sudoku[y][pos] != 0 and y != row:
                return False
        except:
            break

    if not sqr_extract(sudoku, row, pos, num):
        return False

    return True

In [3]:
def check_initial_state(sudoku):
    for n in range(len(sudoku)):
        for i in range(len(sudoku)):
            if not check_constraint(sudoku, n, i):
                return False
    return True

In [4]:

def sudoku_solver(sudoku):
    """
    Solves a Sudoku puzzle and returns its unique solution.

    Input
        sudoku : 9x9 numpy array
            Empty cells are designated by 0.

    Output
        9x9 numpy array of integers
            It contains the solution, if there is one. If there is no solution, all array entries should be -1.
    """

    if not check_initial_state(sudoku):
        return np.array([[-1]*9]*9)

    new_soduku = sudoku.copy()
    frontier = {x: {y: [] for y in range(9)}  for x in range(9)}

    n = 0
    while n < len(new_soduku):

        i = 0
        while i < len(new_soduku[n]):

            check = False
            
            if sudoku[n][i] == 0:
                if len(frontier[n][i]) == 0:
                    nums = np.arange(1,10)
                    for num in nums:
                        check = check_constraint(new_soduku, n, i, num)
                        if check:
                            frontier[n][i].append(num)
                            check = False

                if len(frontier[n][i]) == 0:
                    try:
                        if i == 0:
                            i = 8
                            n -= 1 
                        else:
                            i -= 1

                        if sudoku[n][i] == 0:
                            new_soduku[n][i] = 0
                        while len(frontier[n][i]) < 1:  
                            if i == 0:
                                i = 8
                                n -= 1
                            else:
                                i -= 1

                            if sudoku[n][i] == 0:
                                new_soduku[n][i] = 0
                    except:
                        return np.array([[-1]*9]*9)
                
                else:
                    while True:
                        new_num = np.random.choice(frontier[n][i])
                        if new_num != new_soduku[n][i]:
                            new_soduku[n][i] = new_num
                            frontier[n][i].remove(new_num)
                            break
                    i += 1           
            else:
                i += 1 
        n += 1

    return new_soduku            



## Validation on the provided puzzles

The cell below runs the solver on every puzzle of the four difficulty levels, checks the result against the known solution and reports timings. The hard puzzles can take several seconds each.


In [5]:
import time
import numpy as np

scores = {}
difficulties = ['very_easy', 'easy', 'medium', 'hard']

for difficulty in difficulties:
    sudokus = np.load(f"data/{difficulty}_puzzle.npy")
    solutions = np.load(f"data/{difficulty}_solution.npy")

    count = 0
    total_time = 0.0
    for i in range(len(sudokus)):
        sudoku = sudokus[i].copy()

        start_time = time.process_time()
        solution = sudoku_solver(sudoku)
        elapsed = time.process_time() - start_time
        total_time += elapsed

        if np.array_equal(solution, solutions[i]):
            count += 1
        else:
            print(f"{difficulty} sudoku {i}: wrong solution")

    scores[difficulty] = {'correct': count, 'total': len(sudokus), 'seconds': total_time}
    print(f"{difficulty:>9}: {count}/{len(sudokus)} correct, {total_time:.2f} s in total")


very_easy: 15/15 correct, 0.04 s in total
     easy: 15/15 correct, 0.01 s in total
   medium: 15/15 correct, 0.01 s in total


     hard: 15/15 correct, 26.08 s in total
